# CLIP Text-to-Image Search — Caltech-101
Search a gallery covering **101 object categories** (plus one background category). This replaces CIFAR-10's 10 categories. CLIP is **pretrained**: we only compute image/text embeddings; no training is needed.

Caltech-101 has 9,144 images in total, with varied resolutions. The default selects up to **30 images per class** to keep Colab responsive. Increase `IMAGES_PER_CLASS` to expand the gallery.

In [ ]:
!pip -q install transformers tensorflow-datasets importlib_resources matplotlib pillow

## 1. Load pretrained CLIP

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
print('Device:', DEVICE)

## 2. Download Caltech-101 and build a diverse gallery
The TensorFlow Datasets version has **102 labels**: 101 object types plus background clutter. We skip the background class. Set `IMAGES_PER_CLASS = None` to use **all** object images. Images are resized for display/storage only; CLIP's processor handles model input resizing.

In [ ]:
import tensorflow_datasets as tfds

IMAGES_PER_CLASS = 10  # 30 ≈ 3,000 images; 10 ≈ 1,000; None = all object images.

ds, info = tfds.load('caltech101', split='train+test', with_info=True, shuffle_files=False)
class_names = info.features['label'].names
images, names = [], []
per_class = {name: 0 for name in class_names}

for item in tfds.as_numpy(ds):
    label = class_names[int(item['label'])]
    if 'background' in label.lower():
        continue
    if IMAGES_PER_CLASS is not None and per_class[label] >= IMAGES_PER_CLASS:
        continue
    img = Image.fromarray(item['image']).convert('RGB')
    img.thumbnail((320, 320))  # Keep memory use reasonable.
    images.append(img)
    names.append(label.replace('_', ' '))
    per_class[label] += 1

print('Images:', len(images), '| Categories:', len(set(names)))
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, i in zip(axes.flat, np.random.default_rng(42).choice(len(images), 10, replace=False)):
    ax.imshow(images[i]); ax.set_title(names[i], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Encode images in batches (run once)
Each image becomes a CLIP vector. Encoding the gallery is the slow part; searching new text is fast. GPU is recommended for several thousand images.

In [ ]:
BATCH_SIZE = 128
vectors = []
with torch.no_grad():
    for start in range(0, len(images), BATCH_SIZE):
        batch = processor(images=images[start:start+BATCH_SIZE], return_tensors='pt').to(DEVICE)
        v = model.get_image_features(**batch)
        if hasattr(v, 'pooler_output'):
            v = v.pooler_output
        vectors.append(torch.nn.functional.normalize(v, dim=-1).cpu())
        if start % 640 == 0:
            print(f'Encoded {min(start+BATCH_SIZE, len(images))}/{len(images)} images')
image_vectors = torch.cat(vectors)
print('Embedding matrix:', image_vectors.shape)

## 4. Search with any English phrase
CLIP ranks **only the images in this gallery**. It returns nearest matches even if your description is not present; similarity scores are not probabilities.

In [ ]:
def search_images(query, top_k=6):
    with torch.no_grad():
        text = processor(text=[query], return_tensors='pt', padding=True, truncation=True).to(DEVICE)
        v = model.get_text_features(**text)
        if hasattr(v, 'pooler_output'):
            v = v.pooler_output
        v = torch.nn.functional.normalize(v, dim=-1).cpu()
        scores = (v @ image_vectors.T).squeeze(0).numpy()

    best = np.argsort(-scores)[:min(top_k, len(images))]
    fig, axes = plt.subplots(1, len(best), figsize=(3*len(best), 3.6))
    for ax, i in zip(np.atleast_1d(axes), best):
        ax.imshow(images[i]); ax.set_title(f'{names[i]}\n{scores[i]:.3f}', fontsize=10); ax.axis('off')
    plt.suptitle(query); plt.tight_layout(); plt.show()
    return [(names[i], round(float(scores[i]), 3)) for i in best]

search_images('a motorcycle')
search_images('a musical instrument')

## 5. Search using your own description

In [ ]:
query = input('What kind of picture do you want? ')
search_images(query, top_k=6)

## Student challenges
1. Search for `an animal`, `a vehicle`, and `a musical instrument`.
2. Try synonyms such as `motorbike` vs `motorcycle`.
3. Search for an object absent from the dataset. Does CLIP still return results?
4. Increase `IMAGES_PER_CLASS` from 30 to 60 and rerun Sections 2–5.
5. Explain why the image vectors are computed only once.

**Dataset:** Caltech-101 via TensorFlow Datasets (101 object categories plus background; 9,144 images). **Model:** pretrained `openai/clip-vit-base-patch32`. Results depend on image quality and gallery coverage.